In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
  library(ggpubr)
  library(scales)
  library(patchwork)
  library(viridis)
  library(pheatmap)
  library(readxl)
  library(svglite)
  library(ragg)
  library(grid)
})

# Virtual project paths: replace these with real paths only when reproducing the figures.
PROJECT_DIR <- "/path/to/hippocampal_sclerosis_project"
DATA_DIR    <- file.path(PROJECT_DIR, "data")
RESULT_DIR  <- file.path(PROJECT_DIR, "results")
FIG_DIR     <- file.path(PROJECT_DIR, "figure_exports")
SRC_DIR     <- file.path(PROJECT_DIR, "source_data")

dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(SRC_DIR, recursive = TRUE, showWarnings = FALSE)

GROUP_LEVELS <- c("HS-", "HS+")
GROUP_COLORS <- c("HS-" = "#8DC7C2", "HS+" = "#E94C5F")
DIVERGING_COLORS <- c("#14448C", "#5076C1", "#92A6DE", "#CED7F2",
                      "#F6F6F6", "#F5CECE", "#DE9494", "#B85A5B", "#7D2828")
EXPRESSION_COLORS <- c("#440154", "#31688E", "#35B779", "#FDE725")

theme_pub <- function(base_size = 7) {
  theme_classic(base_size = base_size, base_family = "Arial") +
    theme(
      axis.line = element_line(linewidth = 0.25, colour = "black"),
      axis.ticks = element_line(linewidth = 0.25, colour = "black"),
      axis.text = element_text(colour = "black"),
      legend.key.height = unit(3.5, "mm"),
      legend.key.width = unit(3.5, "mm"),
      legend.title = element_text(size = base_size),
      legend.text = element_text(size = base_size - 1),
      strip.background = element_rect(fill = "grey92", colour = "black", linewidth = 0.25),
      strip.text = element_text(colour = "black", face = "plain")
    )
}
theme_set(theme_pub())

save_pub <- function(plot, file_stub, width_mm, height_mm, dpi = 600) {
  svg_file <- file.path(FIG_DIR, paste0(file_stub, ".svg"))
  pdf_file <- file.path(FIG_DIR, paste0(file_stub, ".pdf"))
  tif_file <- file.path(FIG_DIR, paste0(file_stub, ".tiff"))

  svglite::svglite(svg_file, width = width_mm / 25.4, height = height_mm / 25.4)
  print(plot)
  dev.off()

  cairo_pdf(pdf_file, width = width_mm / 25.4, height = height_mm / 25.4, family = "Arial")
  print(plot)
  dev.off()

  ragg::agg_tiff(tif_file, width = width_mm, height = height_mm, units = "mm",
                 res = dpi, compression = "lzw")
  print(plot)
  dev.off()

  invisible(c(svg = svg_file, pdf = pdf_file, tiff = tif_file))
}

clean_group <- function(x) {
  x <- as.character(x)
  dplyr::case_when(
    x %in% c("HS-", "Normal", "Control", "CTL", "TLE-noHS") ~ "HS-",
    x %in% c("HS+", "HS", "Sclerosis", "TLE-HS") ~ "HS+",
    TRUE ~ x
  )
}

sig_label <- function(p) {
  dplyr::case_when(
    is.na(p) ~ "ns",
    p < 0.0001 ~ "****",
    p < 0.001 ~ "***",
    p < 0.01 ~ "**",
    p < 0.05 ~ "*",
    TRUE ~ "ns"
  )
}

first_existing_col <- function(object, candidates) {
  hit <- candidates[candidates %in% colnames(object@meta.data)]
  if (length(hit) == 0) {
    stop("None of these metadata columns were found: ", paste(candidates, collapse = ", "))
  }
  hit[[1]]
}

plot_score_umap <- function(object, score_col, title, highlight_cluster = NULL,
                            file_stub = NULL, width_mm = 48, height_mm = 42) {
  score_col <- first_existing_col(object, score_col)
  emb <- as.data.frame(Embeddings(object, "umap"))
  colnames(emb)[1:2] <- c("UMAP_1", "UMAP_2")
  emb$score <- object@meta.data[[score_col]]
  emb$subcluster <- object$subcluster

  p <- ggplot(emb, aes(UMAP_1, UMAP_2, colour = score)) +
    geom_point(size = 0.08, stroke = 0, alpha = 0.85) +
    scale_colour_gradientn(colours = EXPRESSION_COLORS, name = "Expression") +
    labs(title = title, x = "UMAP1", y = "UMAP2") +
    coord_equal() +
    theme_pub() +
    theme(
      legend.position = c(0.78, 0.12),
      legend.background = element_blank(),
      axis.text = element_blank(),
      axis.ticks = element_blank()
    )

  if (!is.null(highlight_cluster)) {
    centroid <- emb |>
      filter(subcluster == highlight_cluster) |>
      summarise(x = median(UMAP_1), y = median(UMAP_2))
    p <- p + annotate("text", x = centroid$x, y = centroid$y, label = title,
                      colour = "white", size = 2.2, fontface = "bold")
  }

  if (!is.null(file_stub)) save_pub(p, file_stub, width_mm, height_mm)
  p
}

plot_group_violin <- function(object, cluster, score_col, ylab, file_stub,
                              width_mm = 32, height_mm = 42) {
  score_col <- first_existing_col(object, score_col)
  dat <- object@meta.data |>
    transmute(subcluster, Group, score = .data[[score_col]]) |>
    filter(subcluster == cluster, Group %in% GROUP_LEVELS) |>
    mutate(Group = factor(Group, levels = GROUP_LEVELS))

  pval <- tryCatch(wilcox.test(score ~ Group, data = dat)$p.value, error = function(e) NA_real_)

  p <- ggplot(dat, aes(Group, score, fill = Group)) +
    geom_violin(width = 0.9, linewidth = 0.15, colour = NA, trim = TRUE) +
    geom_boxplot(width = 0.18, linewidth = 0.25, outlier.shape = NA,
                 fill = "white", colour = "grey30") +
    annotate("text", x = 1.5, y = max(dat$score, na.rm = TRUE) * 1.05,
             label = sig_label(pval), size = 2.5) +
    scale_fill_manual(values = GROUP_COLORS) +
    labs(x = NULL, y = ylab, title = paste0(cluster, " in disease axis")) +
    theme_pub() +
    theme(legend.position = "none")

  write.csv(dat, file.path(SRC_DIR, paste0(file_stub, "_source.csv")), row.names = FALSE)
  save_pub(p, file_stub, width_mm, height_mm)
  p
}

make_density_bar <- function(dat, marker_label, file_stub, width_mm = 34, height_mm = 38) {
  plot_dat <- dat |>
    mutate(Group = factor(clean_group(Group), levels = GROUP_LEVELS)) |>
    group_by(Group) |>
    summarise(mean = mean(density_mm2, na.rm = TRUE),
              sem = sd(density_mm2, na.rm = TRUE) / sqrt(dplyr::n()),
              .groups = "drop")
  pval <- tryCatch(wilcox.test(density_mm2 ~ Group, data = dat)$p.value, error = function(e) NA_real_)

  p <- ggplot(plot_dat, aes(Group, mean, fill = Group)) +
    geom_col(width = 0.58, colour = "black", linewidth = 0.25) +
    geom_errorbar(aes(ymin = mean - sem, ymax = mean + sem), width = 0.16, linewidth = 0.25) +
    annotate("text", x = 1.5, y = max(plot_dat$mean + plot_dat$sem, na.rm = TRUE) * 1.12,
             label = sig_label(pval), size = 2.6) +
    scale_fill_manual(values = GROUP_COLORS) +
    labs(x = NULL, y = expression("Cell density (/mm"^2*")"), title = marker_label) +
    theme_pub() +
    theme(legend.position = "none")

  write.csv(dat, file.path(SRC_DIR, paste0(file_stub, "_source.csv")), row.names = FALSE)
  save_pub(p, file_stub, width_mm, height_mm)
  p
}


In [ ]:
ASTRO_COLORS <- c(
  "Astro.0" = "#22B8CF",
  "Astro.1" = "#91A7FF",
  "Astro.2" = "#228BE6",
  "Astro.3" = "#364FC7"
)

astro <- readRDS(file.path(DATA_DIR, "seurat", "Astro_subcluster.rds"))
DefaultAssay(astro) <- "RNA"
astro <- NormalizeData(astro, verbose = FALSE)

# Use curated labels when present. The fallback records the original resolution-based
# mapping used for Figure 4 preparation, without introducing any post-hoc subtype merge.
if (!"subcluster" %in% colnames(astro@meta.data)) {
  cluster_col <- "SCT_snn_res.0.4"
  stopifnot(cluster_col %in% colnames(astro@meta.data))
  astro$subcluster <- dplyr::recode(
    as.character(astro@meta.data[[cluster_col]]),
    "0" = "Astro.0",
    "1" = "Astro.2",
    "2" = "Astro.1",
    "3" = "Astro.1",
    "4" = "Astro.3",
    .default = NA_character_
  )
}

astro$subcluster <- factor(astro$subcluster, levels = names(ASTRO_COLORS))
astro$Group <- factor(clean_group(astro$Group), levels = GROUP_LEVELS)
Idents(astro) <- "subcluster"

write.csv(astro@meta.data, file.path(SRC_DIR, "fig4_astro_metadata.csv"))
table(astro$subcluster, astro$Group)


In [ ]:
# Figure 4a: Astrocyte subtype UMAP.
p4a <- DimPlot(
  astro, reduction = "umap", group.by = "subcluster",
  cols = ASTRO_COLORS, pt.size = 0.08, label = TRUE, repel = TRUE
) +
  labs(title = paste0("n = ", format(ncol(astro), big.mark = ",")),
       x = "UMAP1", y = "UMAP2") +
  coord_equal() +
  theme_pub() +
  theme(axis.text = element_blank(), axis.ticks = element_blank())
save_pub(p4a, "Fig4a_Astro_subcluster_UMAP", 60, 52)
p4a

# Figure 4b: marker confirmation. scale = FALSE keeps the original average

astro_marker_genes <- rev(c("RORA", "GPC5", "SOX6", "AQP1", "SLC1A3", "APOD", "CCL2", "JUN"))
p4b <- DotPlot(astro, features = astro_marker_genes, group.by = "subcluster", scale = FALSE) +
  coord_flip() +
  scale_colour_gradientn(colours = DIVERGING_COLORS, name = "Expression") +
  scale_size(range = c(0.2, 4.2), name = "Ratio") +
  labs(x = NULL, y = NULL) +
  theme_pub() +
  theme(axis.text.x = element_text(angle = 60, hjust = 1),
        panel.border = element_rect(fill = NA, colour = "black", linewidth = 0.25))
save_pub(p4b, "Fig4b_Astro_marker_dotplot", 48, 62)
p4b


In [ ]:

run_marker_enrichment <- FALSE

if (run_marker_enrichment) {
  suppressPackageStartupMessages({
    library(clusterProfiler)
    library(org.Hs.eg.db)
  })

  astro_markers_all <- FindAllMarkers(
    astro,
    only.pos = TRUE,
    test.use = "wilcox",
    min.pct = 0.10,
    logfc.threshold = 0.25
  )

  astro_marker_leaders <- astro_markers_all |>
    filter(p_val_adj < 0.05) |>
    group_by(cluster) |>
    slice_max(avg_log2FC, n = 200, with_ties = FALSE) |>
    ungroup()

  astro_go <- astro_marker_leaders |>
    group_by(cluster) |>
    group_modify(~ {
      entrez <- bitr(.x$gene, fromType = "SYMBOL", toType = "ENTREZID",
                     OrgDb = org.Hs.eg.db) |> pull(ENTREZID) |> unique()
      enrichGO(
        gene = entrez, OrgDb = org.Hs.eg.db, ont = "BP",
        pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.20,
        readable = TRUE
      ) |> as.data.frame()
    }) |>
    ungroup()

  write.csv(astro_markers_all, file.path(SRC_DIR, "fig4_astro_FindAllMarkers.csv"), row.names = FALSE)
  write.csv(astro_go, file.path(SRC_DIR, "fig4_astro_GO_BP_all.csv"), row.names = FALSE)
}


In [ ]:
# Figure 4c: curated BP modules used in the heatmap and downstream UMAP/violin panels.

astro_bp <- readxl::read_xlsx(file.path(DATA_DIR, "curated_gene_sets", "Astro_BP_curated.xlsx")) |>
  transmute(
    module = Description,
    genes = str_split(geneID, "/", simplify = FALSE)
  )

for (i in seq_len(nrow(astro_bp))) {
  astro <- AddModuleScore(
    object = astro,
    features = list(astro_bp$genes[[i]]),
    name = astro_bp$module[[i]],
    assay = "RNA"
  )
}

astro_bp_cols <- paste0(astro_bp$module, "1")
astro_bp_matrix <- astro@meta.data |>
  select(subcluster, all_of(astro_bp_cols)) |>
  group_by(subcluster) |>
  summarise(across(all_of(astro_bp_cols), mean, na.rm = TRUE), .groups = "drop") |>
  tibble::column_to_rownames("subcluster") |>
  as.matrix()

astro_bp_z <- t(scale(t(astro_bp_matrix)))
astro_bp_z[is.na(astro_bp_z)] <- 0

pheatmap(
  astro_bp_z,
  color = colorRampPalette(c("#F6F5EB", "#FEE0D2", "#DE2D26", "#7F1818"))(100),
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  border_color = "grey75",
  fontsize = 7,
  filename = file.path(FIG_DIR, "Fig4c_Astro_BP_heatmap.pdf"),
  width = 3.2,
  height = 2.8
)
write.csv(astro_bp_z, file.path(SRC_DIR, "fig4c_astro_BP_heatmap_zscore.csv"))


In [ ]:
# Figure 4d/e/i/j/n/o: disease-axis modules.

astro_function_panels <- tibble::tribble(
  ~panel, ~cluster,  ~title,                 ~score_candidates,                                        ~region,
  "d/e",  "Astro.0", "SOXD family genes",    list(c("SOXD family genes1", "SOXD.family.genes1")),       "CA",
  "i/j",  "Astro.1", "DAA",                  list(c("DAA1", "Disease associated astrocyte1")),          "CA",
  "n/o",  "Astro.3", "p38 MAPK cascade",     list(c("p38 MAPK cascade1", "p38.MAPK.cascade1")),         "FAS"
)

p4d <- plot_score_umap(astro, astro_function_panels$score_candidates[[1]],
                       "SOXD family genes", "Astro.0", "Fig4d_Astro0_SOXD_UMAP")
p4e <- plot_group_violin(astro, "Astro.0", astro_function_panels$score_candidates[[1]],
                         "SOXD family genes", "Fig4e_Astro0_SOXD_violin")

p4i <- plot_score_umap(astro, astro_function_panels$score_candidates[[2]],
                       "DAA", "Astro.1", "Fig4i_Astro1_DAA_UMAP")
p4j <- plot_group_violin(astro, "Astro.1", astro_function_panels$score_candidates[[2]],
                         "DAA", "Fig4j_Astro1_DAA_violin")

p4n <- plot_score_umap(astro, astro_function_panels$score_candidates[[3]],
                       "p38 MAPK cascade", "Astro.3", "Fig4n_Astro3_p38_UMAP")
p4o <- plot_group_violin(astro, "Astro.3", astro_function_panels$score_candidates[[3]],
                         "p38 MAPK cascade", "Fig4o_Astro3_p38_violin")


In [ ]:
# Figure 4g/l/q: spatial distribution of selected astrocyte states.

spatial_meta <- readRDS(file.path(DATA_DIR, "spatial", "cellbin_spatial_meta.rds"))
spatial_meta <- spatial_meta |>
  mutate(Group = factor(clean_group(Group), levels = GROUP_LEVELS),
         subcluster = factor(subcluster, levels = names(ASTRO_COLORS)))

plot_spatial_state <- function(dat, state, region, file_stub, width_mm = 58, height_mm = 36) {
  plot_dat <- dat |>
    filter(region == !!region, subcluster %in% names(ASTRO_COLORS)) |>
    mutate(is_state = subcluster == state)

  p <- ggplot(plot_dat, aes(x, y)) +
    geom_point(data = filter(plot_dat, !is_state), colour = "grey82", size = 0.05, alpha = 0.35) +
    geom_point(data = filter(plot_dat, is_state), aes(colour = subcluster), size = 0.18, alpha = 0.9) +
    scale_colour_manual(values = ASTRO_COLORS, drop = FALSE) +
    facet_grid(. ~ Group) +
    coord_equal() +
    labs(title = paste0(state, " in ", region), x = NULL, y = NULL) +
    theme_pub() +
    theme(axis.text = element_blank(), axis.ticks = element_blank(), legend.position = "none")

  write.csv(plot_dat, file.path(SRC_DIR, paste0(file_stub, "_source.csv")), row.names = FALSE)
  save_pub(p, file_stub, width_mm, height_mm)
  p
}

p4g <- plot_spatial_state(spatial_meta, "Astro.0", "CA",  "Fig4g_Astro0_CA_spatial")
p4l <- plot_spatial_state(spatial_meta, "Astro.1", "CA",  "Fig4l_Astro1_CA_spatial")
p4q <- plot_spatial_state(spatial_meta, "Astro.3", "FAS", "Fig4q_Astro3_FAS_spatial")


In [ ]:
# Figure 4f/h/k/m/p/r: immunostaining validation statistics.

astro_validation <- read.csv(file.path(DATA_DIR, "validation", "astro_immunostaining_density.csv")) |>
  mutate(
    Group = factor(clean_group(Group), levels = GROUP_LEVELS),
    density_mm2 = positive_count / area_mm2
  )

validation_panels <- tibble::tribble(
  ~marker_set,       ~file_stub,
  "S100B+SOX6",      "Fig4h_S100B_SOX6_density",
  "S100B+AQP1",      "Fig4m_S100B_AQP1_density",
  "S100B+CCL2",      "Fig4r_S100B_CCL2_density"
)

astro_density_plots <- purrr::pmap(
  validation_panels,
  function(marker_set, file_stub) {
    astro_validation |>
      filter(marker_set == !!marker_set) |>
      make_density_bar(marker_set, file_stub)
  }
)


In [ ]:
# Figure 4s: final schematic is assembled in Illustrator/PowerPoint from the

astro_model <- tibble::tribble(
  ~state,              ~dominant_signal,                 ~disease_axis, ~validation,
  "P.astro / Astro.0", "SOXD family genes; Wnt pathway",  "HS+ enriched", "S100B+SOX6 density",
  "R.astro / Astro.1", "DAA; NF-kB regulation",           "HS+ enriched", "S100B+AQP1 density",
  "A.astro / Astro.3", "p38 MAPK cascade; CCL2",          "HS+ enriched", "S100B+CCL2 density"
)
write.csv(astro_model, file.path(SRC_DIR, "fig4s_astro_model_source.csv"), row.names = FALSE)
astro_model
